# Analisis Sentimen Review Aplikasi E-Commerce Indonesia (Play Store)

**Topik:** Review aplikasi Tokopedia, Shopee, dan Lazada di Google Play Store  
**Sumber:** Google Play Store (scraping mandiri via `google-play-scraper`)  
**Pelabelan:** Otomatis dari rating bintang — ★1-2 = Negatif | ★3 = Netral | ★4-5 = Positif

| # | Skema | Algoritma | Ekstraksi Fitur | Split |
|---|-------|-----------|-----------------|-------|
| 1 | Baseline | SVM (LinearSVC) | TF-IDF | 80/20 |
| 2 | Deep Learning | Bi-LSTM | Word2Vec Embedding | 80/20 |
| 3 | Transformer | IndoBERT Fine-tune | Contextual Embedding | 80/20 |

**Target:** Akurasi training & testing > 92% | **Kelas:** Positif · Netral · Negatif

In [ ]:
import subprocess, sys

packages = [
    "google-play-scraper", "pandas", "numpy", "scikit-learn",
    "PySastrawi", "nltk", "matplotlib", "seaborn", "wordcloud",
    "tensorflow", "gensim", "transformers", "torch",
    "sentencepiece", "accelerate", "tqdm", "joblib",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✅ Semua package berhasil diinstall")

In [ ]:
import os, re, warnings, random
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
    SASTRAWI_OK = True
except ImportError:
    SASTRAWI_OK = False
    print('[WARN] Sastrawi tidak tersedia, stemming dilewati.')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from gensim.models import Word2Vec

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback,
)
import torch
from torch.utils.data import Dataset

print('✅ Semua library berhasil diimport')
print(f'   TensorFlow : {tf.__version__}')
print(f'   PyTorch    : {torch.__version__}')
print(f'   GPU        : {torch.cuda.is_available()}')

## 1. Load Dataset

In [ ]:
DATA_PATH = 'data/raw_comments.csv'

if Path(DATA_PATH).exists():
    df = pd.read_csv(DATA_PATH)
    print(f'✅ Dataset dimuat: {DATA_PATH}')
else:
    raise FileNotFoundError(
        f'File {DATA_PATH} tidak ditemukan.\n'
        'Jalankan: python scraping.py'
    )

# Pastikan kolom yang dibutuhkan ada
assert 'text' in df.columns, "Kolom 'text' tidak ditemukan"
assert 'true_label' in df.columns, "Kolom 'true_label' tidak ditemukan"

# Bersihkan
df['true_label'] = df['true_label'].astype(str).str.strip().str.lower()
df = df[df['true_label'].isin(['positif','netral','negatif'])]
df = df[df['text'].notna() & (df['text'].str.strip() != '')]
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f'Shape          : {df.shape}')
print(f'\nDistribusi label:')
for label, cnt in df['true_label'].value_counts().items():
    print(f'  {label:10s}: {cnt:,}')
df.head()

## 2. Eksplorasi Data (EDA)

In [ ]:
df['text_len'] = df['text'].str.len()

print('Statistik panjang teks:')
print(df['text_len'].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribusi label
colors = {'positif':'#2ecc71', 'negatif':'#e74c3c', 'netral':'#3498db'}
vc = df['true_label'].value_counts()
bar_colors = [colors[l] for l in vc.index]
bars = axes[0].bar(vc.index, vc.values, color=bar_colors, edgecolor='white', alpha=0.88)
axes[0].set_title('Distribusi Label Sentimen', fontweight='bold')
axes[0].set_ylabel('Jumlah')
for b, v in zip(bars, vc.values):
    axes[0].text(b.get_x()+b.get_width()/2, v+50, f'{v:,}', ha='center', fontweight='bold', fontsize=9)

# Pie chart
axes[1].pie(vc.values, labels=vc.index, colors=bar_colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporsi Kelas', fontweight='bold')

# Distribusi panjang teks
axes[2].hist(df['text_len'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[2].axvline(df['text_len'].mean(), color='crimson', linestyle='--', label=f"Mean: {df['text_len'].mean():.0f}")
axes[2].set_title('Distribusi Panjang Review', fontweight='bold')
axes[2].set_xlabel('Jumlah Karakter')
axes[2].legend()

plt.tight_layout()
plt.savefig('data/01_eda.png', dpi=120, bbox_inches='tight')
plt.show()

# App distribution jika ada
if 'app_name' in df.columns:
    print('\nDistribusi per aplikasi:')
    for app, cnt in df['app_name'].value_counts().items():
        print(f'  {app:15s}: {cnt:,}')

## 3. Preprocessing Teks

In [ ]:
SLANG = {
    'gk':'tidak','ga':'tidak','gak':'tidak','ngga':'tidak','nggak':'tidak',
    'tdk':'tidak','gpp':'tidak apa','bgt':'banget','bngt':'banget',
    'yg':'yang','dg':'dengan','dgn':'dengan','utk':'untuk','buat':'untuk',
    'sy':'saya','gw':'saya','gue':'saya','lo':'kamu','lu':'kamu',
    'krn':'karena','karna':'karena','kalo':'kalau','tp':'tapi',
    'sdh':'sudah','udh':'sudah','udah':'sudah','blm':'belum',
    'jg':'juga','sm':'sama','dr':'dari','pd':'pada',
    'emg':'memang','emang':'memang','aja':'saja','doang':'saja',
    'gmn':'bagaimana','gimana':'bagaimana','knp':'kenapa',
    'lbh':'lebih','sgt':'sangat','bener':'benar',
    'mantap':'bagus','keren':'bagus','jelek':'buruk','payah':'buruk',
    'makasih':'terima kasih','mksh':'terima kasih','thx':'terima kasih',
    'oke':'baik','ok':'baik','sip':'baik','apk':'aplikasi',
    'app':'aplikasi','update':'pembaruan','bug':'kesalahan',
    'error':'kesalahan','lag':'lambat','lemot':'lambat',
    'promo':'promosi','voucher':'kupon','ongkir':'ongkos kirim',
    'cod':'bayar ditempat','cs':'layanan pelanggan',
}

ID_STOP = set(stopwords.words('indonesian')) | {
    'nya','lah','kah','sih','nih','deh','tuh','dong','woi','wah',
    'hmm','hm','eh','ah','oh','iya','yes','haha','hehe','wkwk',
    'banget','sangat','sekali','benar','memang',
}

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)          # hapus URL
    text = re.sub(r'@\w+|#\w+', '', text)                  # hapus mention/hashtag
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)             # hapus emoji
    text = re.sub(r'[^a-z\s]', ' ', text)                   # hapus non-alpha
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)            # normalisasi huruf berulang
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [SLANG.get(t, t) for t in text.split()]
    tokens = [t for t in tokens if t not in ID_STOP and len(t) > 1]
    if SASTRAWI_OK:
        tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

print('Preprocessing ...')
df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].str.strip().str.len() > 3].reset_index(drop=True)

print(f'✅ Preprocessing selesai — {len(df):,} data tersisa')
print('\nContoh hasil:')
print(df[['text','clean_text']].head(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
wc_cfg = {'positif':'Greens', 'netral':'Blues', 'negatif':'Reds'}

for ax, sentiment in zip(axes, ['positif', 'netral', 'negatif']):
    corpus = ' '.join(df[df['true_label']==sentiment]['clean_text'].dropna())
    if corpus.strip():
        wc = WordCloud(width=400, height=300, background_color='white',
                       colormap=wc_cfg[sentiment], max_words=80,
                       collocations=False).generate(corpus)
        ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'WordCloud — {sentiment.title()}', fontweight='bold', fontsize=13)

plt.suptitle('Kata Dominan per Kelas Sentimen', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('data/02_wordcloud.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Encoding Label & Split Data

In [ ]:
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['true_label'].astype(str))

CLASS_NAMES = [str(c) for c in le.classes_.tolist()]
N_CLASSES   = len(CLASS_NAMES)

print(f'Kelas  : {CLASS_NAMES}')
print(f'Encode : {dict(zip(CLASS_NAMES, le.transform(CLASS_NAMES)))}')

X = df['clean_text'].values
y = df['label_enc'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'\nTrain  : {len(X_train):,}  ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test   : {len(X_test):,}   ({len(X_test)/len(X)*100:.1f}%)')

---
## ⚙️ Skema 1 — SVM + TF-IDF (80/20)

In [ ]:
print('='*55)
print('SKEMA 1 — LinearSVC + TF-IDF (80/20)')
print('='*55)

svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=60_000,
        sublinear_tf=True,
        min_df=2,
    )),
    ('clf', CalibratedClassifierCV(
        LinearSVC(C=1.0, max_iter=3000, class_weight='balanced', random_state=42),
        cv=3,
    )),
])

print('Training ...')
svm_pipeline.fit(X_train, y_train)

y_train_pred_s1 = svm_pipeline.predict(X_train)
y_test_pred_s1  = svm_pipeline.predict(X_test)
acc_train_s1    = accuracy_score(y_train, y_train_pred_s1)
acc_test_s1     = accuracy_score(y_test,  y_test_pred_s1)

print(f'\n✅ Skema 1 — SVM + TF-IDF')
print(f'   Training Accuracy : {acc_train_s1*100:.2f}%')
print(f'   Testing  Accuracy : {acc_test_s1*100:.2f}%')
print()
print(classification_report(y_test, y_test_pred_s1, target_names=CLASS_NAMES))

os.makedirs('models', exist_ok=True)
joblib.dump(svm_pipeline, 'models/skema1_svm_tfidf.pkl')
print('✅ Model disimpan → models/skema1_svm_tfidf.pkl')

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_test_pred_s1), display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Skema 1 — SVM + TF-IDF\nTest Acc: {acc_test_s1*100:.2f}%', fontweight='bold')
plt.tight_layout()
plt.savefig('data/03_cm_skema1.png', dpi=120, bbox_inches='tight')
plt.show()

---
## ⚙️ Skema 2 — Bi-LSTM + Word2Vec (80/20)

In [ ]:
print('='*55)
print('SKEMA 2 — Bi-LSTM + Word2Vec Embedding (80/20)')
print('='*55)

MAX_VOCAB  = 40_000
MAX_LEN    = 120
EMBED_DIM  = 128
BATCH_SIZE = 64
EPOCHS     = 20

# Tokenisasi Keras
keras_tok = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
keras_tok.fit_on_texts(X_train)
X_tr_seq = pad_sequences(keras_tok.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_te_seq = pad_sequences(keras_tok.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')
print(f'Vocab size  : {len(keras_tok.word_index):,}')

# Word2Vec
print('Melatih Word2Vec ...')
w2v = Word2Vec([t.split() for t in X], vector_size=EMBED_DIM, window=5,
               min_count=2, workers=4, sg=1, epochs=15)

embed_matrix = np.zeros((MAX_VOCAB+1, EMBED_DIM))
hit = 0
for word, idx in keras_tok.word_index.items():
    if idx <= MAX_VOCAB and word in w2v.wv:
        embed_matrix[idx] = w2v.wv[word]
        hit += 1
print(f'Embedding coverage: {hit/min(len(keras_tok.word_index),MAX_VOCAB)*100:.1f}%')

# Arsitektur Bi-LSTM
tf.keras.backend.clear_session()
tf.random.set_seed(42)
inp = layers.Input(shape=(MAX_LEN,))
x   = layers.Embedding(MAX_VOCAB+1, EMBED_DIM, weights=[embed_matrix],
                        input_length=MAX_LEN, trainable=True)(inp)
x   = layers.SpatialDropout1D(0.25)(x)
x   = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
x   = layers.Bidirectional(layers.LSTM(64))(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.4)(x)
x   = layers.Dense(64, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(N_CLASSES, activation='softmax')(x)

lstm_model = keras.Model(inp, out, name='BiLSTM')
lstm_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = dict(enumerate(cw))

history2 = lstm_model.fit(
    X_tr_seq, y_train,
    validation_data = (X_te_seq, y_test),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = cw_dict,
    callbacks       = [
        EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ],
    verbose=1,
)

In [ ]:
_, acc_train_s2 = lstm_model.evaluate(X_tr_seq, y_train, verbose=0)
_, acc_test_s2  = lstm_model.evaluate(X_te_seq, y_test,  verbose=0)
y_test_pred_s2  = np.argmax(lstm_model.predict(X_te_seq, verbose=0), axis=1)

print(f'\n✅ Skema 2 — Bi-LSTM + Word2Vec')
print(f'   Training Accuracy : {acc_train_s2*100:.2f}%')
print(f'   Testing  Accuracy : {acc_test_s2*100:.2f}%')
print()
print(classification_report(y_test, y_test_pred_s2, target_names=CLASS_NAMES))
lstm_model.save('models/skema2_bilstm.keras')
print('✅ Model disimpan → models/skema2_bilstm.keras')

# Plot history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
    ax.plot(history2.history[metric],     label='Train', color='steelblue')
    ax.plot(history2.history[f'val_{metric}'], label='Val', color='crimson')
    ax.set_title(f'Skema 2 — {title}', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('data/04_history_skema2.png', dpi=120, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_test_pred_s2), display_labels=CLASS_NAMES).plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title(f'Skema 2 — Bi-LSTM + Word2Vec\nTest Acc: {acc_test_s2*100:.2f}%', fontweight='bold')
plt.tight_layout()
plt.savefig('data/05_cm_skema2.png', dpi=120, bbox_inches='tight')
plt.show()

---
## ⚙️ Skema 3 — IndoBERT Fine-tuning (80/20)

In [ ]:
print('='*55)
print('SKEMA 3 — IndoBERT Fine-tuning (80/20)')
print('='*55)

BERT_NAME       = 'indobenchmark/indobert-base-p1'
BERT_MAX_LEN    = 128
BERT_BATCH_SIZE = 32
BERT_EPOCHS     = 5
BERT_LR         = 2e-5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

print(f'Loading {BERT_NAME} ...')
bert_tok   = AutoTokenizer.from_pretrained(BERT_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_NAME, num_labels=N_CLASSES, ignore_mismatched_sizes=True)
print('✅ IndoBERT berhasil dimuat')

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.enc    = tokenizer(list(texts), truncation=True, padding=True,
                                max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):  return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item

train_ds = ReviewDataset(X_train, y_train, bert_tok, BERT_MAX_LEN)
test_ds  = ReviewDataset(X_test,  y_test,  bert_tok, BERT_MAX_LEN)
print(f'Train: {len(train_ds):,}  |  Test: {len(test_ds):,}')

In [ ]:
def compute_metrics(ep):
    preds  = np.argmax(ep.predictions, axis=1)
    return {'accuracy': accuracy_score(ep.label_ids, preds)}

training_args = TrainingArguments(
    output_dir                  = 'models/bert_ckpt',
    num_train_epochs            = BERT_EPOCHS,
    per_device_train_batch_size = BERT_BATCH_SIZE,
    per_device_eval_batch_size  = BERT_BATCH_SIZE,
    learning_rate               = BERT_LR,
    weight_decay                = 0.01,
    warmup_steps                = 200,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'accuracy',
    logging_steps               = 100,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
    seed                        = 42,
)

trainer = Trainer(
    model           = bert_model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = test_ds,
    data_collator   = DataCollatorWithPadding(tokenizer=bert_tok),
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Memulai fine-tuning IndoBERT ...')
trainer.train()
print('✅ Fine-tuning selesai!')

In [ ]:
train_out = trainer.predict(train_ds)
test_out  = trainer.predict(test_ds)

y_train_pred_s3 = np.argmax(train_out.predictions, axis=1)
y_test_pred_s3  = np.argmax(test_out.predictions,  axis=1)
acc_train_s3    = accuracy_score(y_train, y_train_pred_s3)
acc_test_s3     = accuracy_score(y_test,  y_test_pred_s3)

print(f'\n✅ Skema 3 — IndoBERT Fine-tune')
print(f'   Training Accuracy : {acc_train_s3*100:.2f}%')
print(f'   Testing  Accuracy : {acc_test_s3*100:.2f}%')
print()
print(classification_report(y_test, y_test_pred_s3, target_names=CLASS_NAMES))

trainer.save_model('models/skema3_indobert')
bert_tok.save_pretrained('models/skema3_indobert')
print('✅ Model disimpan → models/skema3_indobert/')

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_test_pred_s3), display_labels=CLASS_NAMES).plot(ax=ax, cmap='Oranges', colorbar=False)
ax.set_title(f'Skema 3 — IndoBERT\nTest Acc: {acc_test_s3*100:.2f}%', fontweight='bold')
plt.tight_layout()
plt.savefig('data/06_cm_skema3.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Perbandingan Semua Skema

In [ ]:
results = pd.DataFrame({
    'Skema'         : ['Skema 1\nSVM+TF-IDF','Skema 2\nBi-LSTM+W2V','Skema 3\nIndoBERT'],
    'Algoritma'     : ['LinearSVC','Bi-LSTM','IndoBERT'],
    'Fitur'         : ['TF-IDF','Word2Vec','Contextual'],
    'Split'         : ['80/20','80/20','80/20'],
    'Train Acc (%)'   : [round(acc_train_s1*100,2), round(acc_train_s2*100,2), round(acc_train_s3*100,2)],
    'Test Acc (%)'    : [round(acc_test_s1*100,2),  round(acc_test_s2*100,2),  round(acc_test_s3*100,2)],
})

print('='*65)
print('RINGKASAN 3 SKEMA PELATIHAN')
print('='*65)
print(results.to_string(index=False))
print('='*65)

fig, ax = plt.subplots(figsize=(10,5))
x = np.arange(3); w = 0.35
b1 = ax.bar(x-w/2, results['Train Acc (%)'], w, label='Train', color='steelblue', alpha=0.85)
b2 = ax.bar(x+w/2, results['Test Acc (%)'],  w, label='Test',  color='coral',     alpha=0.85)
ax.axhline(85, color='orange', linestyle='--', lw=1.5, label='Target 85%')
ax.axhline(92, color='red',    linestyle='--', lw=1.5, label='Target 92%')
ax.set_xticks(x)
ax.set_xticklabels(['Skema 1\nSVM+TF-IDF','Skema 2\nBi-LSTM+W2V','Skema 3\nIndoBERT'])
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,105)
ax.set_title('Perbandingan Akurasi 3 Skema', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for b in list(b1)+list(b2):
    ax.annotate(f'{b.get_height():.1f}%',
                xy=(b.get_x()+b.get_width()/2, b.get_height()),
                xytext=(0,3), textcoords='offset points', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('data/07_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Inference — Prediksi Sentimen Baru

In [ ]:
BEST_MODEL = 'bert'   # ganti ke 'svm' atau 'lstm' jika BERT belum selesai

def predict(text: str, model: str = BEST_MODEL) -> dict:
    clean = clean_text(text)
    if model == 'svm':
        pred  = svm_pipeline.predict([clean])[0]
        proba = svm_pipeline.predict_proba([clean])[0]
        label = le.inverse_transform([pred])[0]
        conf  = proba.max()
    elif model == 'lstm':
        seq   = pad_sequences(keras_tok.texts_to_sequences([clean]), maxlen=MAX_LEN, padding='post')
        proba = lstm_model.predict(seq, verbose=0)[0]
        label = le.inverse_transform([np.argmax(proba)])[0]
        conf  = proba.max()
    else:  # bert
        enc = bert_tok(clean, return_tensors='pt', truncation=True,
                       max_length=128, padding=True)
        enc = {k: v.to(device) for k, v in enc.items()}
        bert_model.eval()
        with torch.no_grad():
            logits = bert_model(**enc).logits
        proba = torch.softmax(logits, dim=1).cpu().numpy()[0]
        label = le.inverse_transform([np.argmax(proba)])[0]
        conf  = proba.max()
    return {'text': text, 'clean': clean, 'sentiment': label.upper(), 'confidence': f'{conf*100:.2f}%'}

# Contoh inferensi
SAMPLES = [
    # Positif
    'Tokopedia aplikasinya sangat mudah digunakan, pengiriman cepat dan aman!',
    'Shopee terbaik deh, banyak promo voucher gratis ongkir setiap hari!',
    'Belanja di Lazada sudah bertahun-tahun, produk original semua terpercaya!',
    # Negatif
    'Aplikasi Tokopedia sering crash dan lemot sekali, sangat mengecewakan!',
    'Shopee customer service tidak membantu sama sekali saat ada masalah!',
    'Lazada barang yang datang tidak sesuai foto, komplain tidak direspon!',
    # Netral
    'Tokopedia biasa saja, sama seperti marketplace lain yang ada di Indonesia.',
    'Shopee lumayan tapi perlu perbaikan lagi di fitur pencarian produknya.',
    'Lazada ada kelebihan dan kekurangannya, tergantung kebutuhan masing-masing.',
]

EMOJI = {'POSITIF':'😊 ✅', 'NEGATIF':'😞 ❌', 'NETRAL':'😐 ➖'}

print('='*65)
print('HASIL INFERENSI MODEL SENTIMEN REVIEW E-COMMERCE')
print('='*65)
for sample in SAMPLES:
    r = predict(sample)
    emoji = EMOJI.get(r['sentiment'], '')
    print(f"Input : {r['text'][:70]}{'...' if len(r['text'])>70 else ''}")
    print(f"Hasil : {emoji} {r['sentiment']}  (confidence: {r['confidence']})")
    print('-'*65)

In [ ]:
# Uji kustom
CUSTOM = [
    'Proses refund Tokopedia sangat lambat, sudah seminggu uang tidak kembali!',
    'Shopee Coins hemat banget, setiap belanja dapat cashback lumayan banyak!',
    'Lazada pengiriman standar, tidak ada yang spesial tapi tidak mengecewakan.',
]

print('\n' + '='*65)
print('UJI TEKS KUSTOM')
print('='*65)
for text in CUSTOM:
    r = predict(text)
    print(f"\n📝 Input      : {r['text']}")
    print(f"🔍 Cleaned    : {r['clean']}")
    print(f"🎯 Sentimen   : {EMOJI.get(r['sentiment'],'')} {r['sentiment']}")
    print(f"📊 Confidence : {r['confidence']}")

## 7. Simpan & Ringkasan Akhir

In [ ]:
joblib.dump(le, 'models/label_encoder.pkl')
df[['text','clean_text','true_label']].to_csv('data/labeled_reviews.csv', index=False, encoding='utf-8-sig')

print('\n' + '='*65)
print('RINGKASAN PROYEK ANALISIS SENTIMEN')
print('='*65)
print(f'Topik        : Review Aplikasi E-Commerce (Tokopedia/Shopee/Lazada)')
print(f'Sumber       : Google Play Store')
print(f'Total Data   : {len(df):,} review')
print(f'Kelas        : {CLASS_NAMES}')
print(f'Pelabelan    : Otomatis dari rating bintang (★1-2=negatif, ★3=netral, ★4-5=positif)')
print(f'Split        : 80% Train / 20% Test')
print()
print(f'Skema 1 — SVM + TF-IDF        : Train {acc_train_s1*100:.2f}%  |  Test {acc_test_s1*100:.2f}%')
print(f'Skema 2 — Bi-LSTM + Word2Vec   : Train {acc_train_s2*100:.2f}%  |  Test {acc_test_s2*100:.2f}%')
print(f'Skema 3 — IndoBERT Fine-tune   : Train {acc_train_s3*100:.2f}%  |  Test {acc_test_s3*100:.2f}%')
best = results.loc[results['Test Acc (%)'].idxmax()]
print()
print(f'🏆 Model Terbaik : {best["Algoritma"]} (Test Acc: {best["Test Acc (%)"]:.2f}%)')
print('='*65)